# ML-07 — Baseline Action Score and Top-10 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

I am keeping Lane 4: CTR/engagement opportunity scoring.

A page is worth a title/meta rewrite if it gets enough impressions to trust.
It must rank well enough that people see it.
Its CTR must be below what other pages at the same position tier get.

Reason code: `ctr_below_peer_at_position`

Action label: `REWRITE_TITLE_META`

In [1]:
from pathlib import Path

import pandas as pd

data_file = Path("data/raw/content_refresh_anonymized.csv")
if not data_file.is_file():
    data_file = next(
        (parent / "data/raw/content_refresh_anonymized.csv" for parent in Path.cwd().parents
         if (parent / "data/raw/content_refresh_anonymized.csv").is_file()),
        None,
    )
if data_file is None:
    raise FileNotFoundError("Could not find data/raw/content_refresh_anonymized.csv from this folder or its parents.")

df = pd.read_csv(data_file)
eligible = df.loc[
    (df["impressions_90d"] >= 500)
    & df["ctr"].notna()
    & df["position_tier"].notna()
].copy()

tier_order = ["top_3", "page_1", "striking", "page_3_5", "deep"]
bucket_table = (
    eligible.groupby("position_tier")["ctr"]
    .agg(n="count", median_ctr="median", mean_ctr="mean")
    .reindex(tier_order)
)
bucket_table["n"] = bucket_table["n"].fillna(0).astype(int)
print(bucket_table.to_string())

                  n  median_ctr  mean_ctr
position_tier                            
top_3           458        0.20  0.346572
page_1         7064        0.24  0.338808
striking       4485        0.17  0.266798
page_3_5       4330        0.09  0.143236
deep            389        0.00  0.043213


**MIXED**

I see the median CTR rise from 0.20 in top_3 to 0.24 in page_1, then fall to 0.17, 0.09, and 0.00, so it does not fall at every tier. I see the mean fall from 0.35 to 0.04, but top_3 (n=458) and deep (n=389) have small n and are noisy.

In [2]:
signal2 = eligible.loc[eligible["days_since_last_update"].notna()].copy()
signal2["position_peer_median_ctr"] = signal2.groupby("position_tier")["ctr"].transform("median")
signal2["ctr_minus_position_peer_median"] = (
    signal2["ctr"] - signal2["position_peer_median_ctr"]
)

update_bucket_order = ["0-90", "91-180", "181-365", "365+"]
signal2["update_bucket"] = pd.cut(
    signal2["days_since_last_update"],
    bins=[0, 90, 180, 365, float("inf")],
    labels=update_bucket_order,
    include_lowest=True,
)
update_bucket_table = (
    signal2.groupby("update_bucket", observed=False)["ctr_minus_position_peer_median"]
    .agg(n="count", median_peer_ctr_difference="median")
    .reindex(update_bucket_order)
)
update_bucket_table["n"] = update_bucket_table["n"].fillna(0).astype(int)
print(update_bucket_table.to_string())

                   n  median_peer_ctr_difference
update_bucket                                   
0-90           10151                        0.00
91-180          6558                        0.00
181-365           17                        0.04
365+               0                         NaN


**MIXED**

I see no lower median peer-adjusted CTR for older pages: the 0-90 and 91-180 buckets are at 0.00, while 181-365 is +0.04 and 365+ has no rows. This does not support staleness, so I will drop it from my score.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [3]:
queue_columns = [
    "content_id",
    "client_id",
    "position_tier",
    "impressions_90d",
    "ctr",
]
queue = df[queue_columns].copy()
queue_eligible = (
    (queue["impressions_90d"] >= 500)
    & queue["ctr"].notna()
    & queue["position_tier"].notna()
)
ctr_benchmarks = eligible.groupby("position_tier")["ctr"].median()
queue["ctr_bench"] = queue["position_tier"].map(ctr_benchmarks)

queue["score"] = 0.0
scored_rows = queue_eligible & queue["ctr_bench"].notna()
ctr_gap = queue.loc[scored_rows, "ctr_bench"] - queue.loc[scored_rows, "ctr"]
queue.loc[scored_rows, "score"] = (
    ctr_gap.clip(lower=0) * queue.loc[scored_rows, "impressions_90d"] / 100
)
queue["reason_code"] = "NO_ACTION"
queue["action"] = "NO_ACTION"
has_opportunity = queue["score"] > 0
queue.loc[has_opportunity, "reason_code"] = "ctr_below_peer_at_position"
queue.loc[has_opportunity, "action"] = "REWRITE_TITLE_META"

queue = queue.sort_values(
    ["score", "content_id"], ascending=[False, True], kind="mergesort"
).reset_index(drop=True)
queue["rank"] = range(1, len(queue) + 1)
queue_columns = [
    "content_id",
    "client_id",
    "position_tier",
    "impressions_90d",
    "ctr",
    "ctr_bench",
    "score",
    "reason_code",
    "action",
    "rank",
]
queue = queue[queue_columns]

output_dir = data_file.resolve().parents[2] / "work" / "outputs"
output_dir.mkdir(parents=True, exist_ok=True)
output_file = output_dir / "baseline_action_score.csv"
queue.to_csv(output_file, index=False)
print("Shape:", queue.shape)
display_columns = [column for column in queue.columns if column != "client_id"]
print("Shape:", queue.shape)
print(queue.loc[:, display_columns].head(10).to_string(index=False))

from datetime import date
import json

metrics = {
    "n_eligible": int(queue_eligible.sum()),
    "n_flagged": int((queue["score"] > 0).sum()),
    "signal_1_verdict": "MIXED",
    "signal_2_verdict": "MIXED",
    "top_10_total_expected_clicks_lost": float(queue.head(10)["score"].sum()),
    "run_date": date.today().isoformat(),
}
metrics_file = output_dir / "w04_baseline_metrics.json"
with metrics_file.open("w", encoding="utf-8") as file:
    json.dump(metrics, file, indent=2)
    file.write("\n")
print(json.dumps(metrics, indent=2))

Shape: (30000, 10)
Shape: (30000, 10)
          content_id position_tier  impressions_90d  ctr  ctr_bench    score                reason_code             action  rank
content_36ff89c8214e        page_1           295097 0.05       0.24 560.6843 ctr_below_peer_at_position REWRITE_TITLE_META     1
content_5fe46e04994d        page_1           517715 0.14       0.24 517.7150 ctr_below_peer_at_position REWRITE_TITLE_META     2
content_c8e9d6ab9013        page_1           208678 0.00       0.24 500.8272 ctr_below_peer_at_position REWRITE_TITLE_META     3
content_c84a0ab98e90        page_1           223271 0.03       0.24 468.8691 ctr_below_peer_at_position REWRITE_TITLE_META     4
content_8451fc6f034d         top_3           272144 0.03       0.20 462.6448 ctr_below_peer_at_position REWRITE_TITLE_META     5
content_453722754fea        page_1           140079 0.01       0.24 322.1817 ctr_below_peer_at_position REWRITE_TITLE_META     6
content_73c54f78c06a        page_1           213963 0.10   

## 3. Top-10 review

In [4]:
display_columns = [column for column in queue.columns if column != "client_id"]
print(queue.loc[:, display_columns].head(10).to_string(index=False))

          content_id position_tier  impressions_90d  ctr  ctr_bench    score                reason_code             action  rank
content_36ff89c8214e        page_1           295097 0.05       0.24 560.6843 ctr_below_peer_at_position REWRITE_TITLE_META     1
content_5fe46e04994d        page_1           517715 0.14       0.24 517.7150 ctr_below_peer_at_position REWRITE_TITLE_META     2
content_c8e9d6ab9013        page_1           208678 0.00       0.24 500.8272 ctr_below_peer_at_position REWRITE_TITLE_META     3
content_c84a0ab98e90        page_1           223271 0.03       0.24 468.8691 ctr_below_peer_at_position REWRITE_TITLE_META     4
content_8451fc6f034d         top_3           272144 0.03       0.20 462.6448 ctr_below_peer_at_position REWRITE_TITLE_META     5
content_453722754fea        page_1           140079 0.01       0.24 322.1817 ctr_below_peer_at_position REWRITE_TITLE_META     6
content_73c54f78c06a        page_1           213963 0.10       0.24 299.5482 ctr_below_peer_at_po

#1 content_36ff89c8214e: action = REWRITE_TITLE_META; why it's here: 295,097 impressions, CTR 0.05%, ctr_bench 0.24%, position_tier page_1; what would make it wrong: Its average position is 7.3, but some queries may rank much lower.
#2 content_5fe46e04994d: action = REWRITE_TITLE_META; why it's here: 517,715 impressions, CTR 0.14%, ctr_bench 0.24%, position_tier page_1; what would make it wrong: Its average position of 4.2 is close to the top_3 cutoff, so its rank may explain some of the gap.
#3 content_c8e9d6ab9013: action = REWRITE_TITLE_META; why it's here: 208,678 impressions, CTR 0.00%, ctr_bench 0.24%, position_tier page_1; what would make it wrong: Its average position of 9.7 is close to the page_1 cutoff at 10.
#4 content_c84a0ab98e90: action = REWRITE_TITLE_META; why it's here: 223,271 impressions, CTR 0.03%, ctr_bench 0.24%, position_tier page_1; what would make it wrong: Informational searches may answer the reader's question in the result, even for this keyword article.
#5 content_8451fc6f034d: action = REWRITE_TITLE_META; why it's here: 272,144 impressions, CTR 0.03%, ctr_bench 0.20%, position_tier top_3; what would make it wrong: The top_3 benchmark has only 458 eligible pages, so it may be noisy.
#6 content_453722754fea: action = REWRITE_TITLE_META; why it's here: 140,079 impressions, CTR 0.01%, ctr_bench 0.24%, position_tier page_1; what would make it wrong: A few high-impression queries with low click intent could pull down its rate.
#7 content_73c54f78c06a: action = REWRITE_TITLE_META; why it's here: 213,963 impressions, CTR 0.10%, ctr_bench 0.24%, position_tier page_1; what would make it wrong: Its 90-day CTR could hide one brief dip that has already passed.
#8 content_91652435f57a: action = REWRITE_TITLE_META; why it's here: 159,590 impressions, CTR 0.06%, ctr_bench 0.24%, position_tier page_1; what would make it wrong: A result feature may show enough information that searchers do not click.
#9 content_c1fe78bc4e37: action = REWRITE_TITLE_META; why it's here: 134,055 impressions, CTR 0.03%, ctr_bench 0.24%, position_tier page_1; what would make it wrong: Its average position of 7.5 could hide a wide spread in query rankings.
#10 content_0919dd345d80: action = REWRITE_TITLE_META; why it's here: 119,217 impressions, CTR 0.02%, ctr_bench 0.24%, position_tier page_1; what would make it wrong: Its 90-day impressions may be concentrated in queries with unusually low click rates.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [5]:
top10_context = queue.head(10).merge(
    df[["content_id", "content_type"]],
    on="content_id",
    how="left",
    validate="one_to_one",
)
tier_counts = eligible.groupby("position_tier").size()
top10_context["tier_n"] = top10_context["position_tier"].map(tier_counts)
weakest_picks = top10_context.loc[top10_context["tier_n"] < 1000]

print("Weakest top-10 picks (position tier has fewer than 1,000 eligible pages):")
print(
    weakest_picks[
        ["rank", "content_id", "position_tier", "tier_n", "impressions_90d", "content_type"]
    ].to_string(index=False)
)

score_input_columns = ["impressions_90d", "ctr", "position_tier", "ctr_bench"]
print("Score input columns:", score_input_columns)
forbidden_terms = ["trend", "last_30d", "prev_30d", "health", "label"]
assert not any(
    term in column.lower()
    for column in score_input_columns
    for term in forbidden_terms
), "A forbidden column appears in the score inputs."
print("Leakage check passed: no forbidden input columns.")

Weakest top-10 picks (position tier has fewer than 1,000 eligible pages):


 rank           content_id position_tier  tier_n  impressions_90d    content_type
    5 content_8451fc6f034d         top_3     458           272144 keyword article
Score input columns: ['impressions_90d', 'ctr', 'position_tier', 'ctr_bench']
Leakage check passed: no forbidden input columns.


**My weakest pick is #5, the top_3 page: its peer tier has only 458 eligible pages, though this page has 272,144 impressions. I would require at least 1,000 eligible pages in a position tier before using its median CTR, and skip scoring tiers below that count.**

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.